In [1]:
import os
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix

class ShopperSpectrumPipeline:
    def __init__(self, filepath='online_retail.csv'):
        self.filepath = filepath
        self.df = None
        self.rfm = None
        self.scaler = StandardScaler()
        self.kmeans = None
        self.cluster_mapping = {}
        self.item_similarity_df = None

    def load_and_clean_data(self):
        print("📥 Step 1: Ingesting and sanitizing transaction logging matrix...")
        if not os.path.exists(self.filepath):
            raise FileNotFoundError(f"Missing essential dataset file: {self.filepath}")
        
        self.df = pd.read_csv(self.filepath)
        
        # Drop entries missing vital categorical identifiers
        self.df = self.df.dropna(subset=['CustomerID', 'Description'])
        self.df['CustomerID'] = self.df['CustomerID'].astype(int)
        
        # Eliminate transaction corrections/cancellations and numerical anomalies
        self.df = self.df[~self.df['InvoiceNo'].astype(str).str.startswith('C')]
        self.df = self.df[(self.df['Quantity'] > 0) & (self.df['UnitPrice'] > 0)]
        
        # Formalize temporal vectors and compound transaction volume values
        self.df['InvoiceDate'] = pd.to_datetime(self.df['InvoiceDate'])
        self.df['TotalAmount'] = self.df['Quantity'] * self.df['UnitPrice']
        print(f"✔️ Cleaning absolute. Sanitized dataset footprint shape: {self.df.shape}")

    def engineer_rfm_features(self):
        print("\n📈 Step 2: Extracting RFM behavioral tensor dimensions...")
        snapshot_date = self.df['InvoiceDate'].max() + pd.Timedelta(days=1)
        
        self.rfm = self.df.groupby('CustomerID').agg({
            'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
            'InvoiceNo': 'nunique',
            'TotalAmount': 'sum'
        }).reset_index()
        
        self.rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']
        print(f"✔️ Extracted behavioral arrays for {self.rfm.shape[0]} unique customer entities.")

    def fit_customer_segmentation(self, n_clusters=4):
        print("\n🧮 Step 3: Calibrating K-Means segmentation clusters...")
        X = self.rfm[['Recency', 'Frequency', 'Monetary']]
        
        # Counteracting extreme power-law skews using continuous log transformation
        X_log = np.log1p(X)
        X_scaled = self.scaler.fit_transform(X_log)
        
        self.kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=15)
        self.rfm['Cluster'] = self.kmeans.fit_predict(X_scaled)
        
        # DYNAMIC PROFILE IDENTIFICATION: Map clusters mathematically
        cluster_means = self.rfm.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].mean()
        
        # Rank by mathematical attributes to completely isolate labels safely
        highest_spending_idx = cluster_means['Monetary'].idxmax()
        oldest_recency_idx = cluster_means['Recency'].idxmax()
        
        remaining_indices = [i for i in range(n_clusters) if i not in [highest_spending_idx, oldest_recency_idx]]
        
        # Sort remaining clusters by order frequency
        if cluster_means.loc[remaining_indices[0], 'Frequency'] > cluster_means.loc[remaining_indices[1], 'Frequency']:
            regular_idx, occasional_idx = remaining_indices[0], remaining_indices[1]
        else:
            regular_idx, occasional_idx = remaining_indices[1], remaining_indices[0]
            
        self.cluster_mapping = {
            highest_spending_idx: "High-Value",
            oldest_recency_idx: "At-Risk",
            regular_idx: "Regular",
            occasional_idx: "Occasional"
        }
        
        self.rfm['Segment'] = self.rfm['Cluster'].map(self.cluster_mapping)
        print("✔️ Segment distribution matrix calculated successfully:")
        for k, v in self.cluster_mapping.items():
            count = (self.rfm['Segment'] == v).sum()
            print(f"  ▪️ Cluster #{k} ➡️ {v:12} (Size: {count})")

    def build_collaborative_filter(self):
        print("\n🛒 Step 4: Compiling compressed sparse collaborative filtering indexes...")
        # Strip product spaces down to structural category indices to protect host RAM
        item_user_matrix = self.df.groupby(['Description', 'CustomerID'])['Quantity'].sum().reset_index()
        
        item_user_matrix['Description'] = item_user_matrix['Description'].astype('category')
        item_user_matrix['CustomerID'] = item_user_matrix['CustomerID'].astype('category')
        
        # Parse interaction arrays securely inside a CSR compressed framework
        sparse_interactions = csr_matrix((
            item_user_matrix['Quantity'],
            (item_user_matrix['Description'].cat.codes, item_user_matrix['CustomerID'].cat.codes)
        ))
        
        similarities = cosine_similarity(sparse_interactions, dense_output=True)
        categories = item_user_matrix['Description'].cat.categories
        
        self.item_similarity_df = pd.DataFrame(similarities, index=categories, columns=categories)
        print(f"✔️ Synthesized product matrix lookup layout. Boundaries: {self.item_similarity_df.shape}")

    def save_pipeline_artifacts(self):
        print("\n💾 Step 5: Serializing analytical models to asset subfolders...")
        os.makedirs('models', exist_ok=True)
        
        artifacts = {
            'scaler.pkl': self.scaler,
            'kmeans.pkl': self.kmeans,
            'cluster_mapping.pkl': self.cluster_mapping,
            'item_similarity_df.pkl': self.item_similarity_df
        }
        
        for filename, obj in artifacts.items():
            with open(f'models/{filename}', 'wb') as f:
                pickle.dump(obj, f)
        print("🎉 Pipeline executed cleanly! Deployment assets exported with zero defects.")

if __name__ == '__main__':
    pipeline = ShopperSpectrumPipeline()
    pipeline.load_and_clean_data()
    pipeline.engineer_rfm_features()
    pipeline.fit_customer_segmentation()
    pipeline.build_collaborative_filter()
    pipeline.save_pipeline_artifacts()

📥 Step 1: Ingesting and sanitizing transaction logging matrix...
✔️ Cleaning absolute. Sanitized dataset footprint shape: (397884, 9)

📈 Step 2: Extracting RFM behavioral tensor dimensions...
✔️ Extracted behavioral arrays for 4338 unique customer entities.

🧮 Step 3: Calibrating K-Means segmentation clusters...
✔️ Segment distribution matrix calculated successfully:
  ▪️ Cluster #3 ➡️ High-Value   (Size: 705)
  ▪️ Cluster #0 ➡️ At-Risk      (Size: 1617)
  ▪️ Cluster #1 ➡️ Regular      (Size: 1176)
  ▪️ Cluster #2 ➡️ Occasional   (Size: 840)

🛒 Step 4: Compiling compressed sparse collaborative filtering indexes...
✔️ Synthesized product matrix lookup layout. Boundaries: (3877, 3877)

💾 Step 5: Serializing analytical models to asset subfolders...
🎉 Pipeline executed cleanly! Deployment assets exported with zero defects.
